# Baggage Threat Localization – Binary U-Net (PyTorch)

Trains a binary U-Net (background vs. threat) on X-ray baggage images, evaluates with Dice / IoU,
post-processes predicted masks, and extracts bounding boxes via connected components.

**Before running:** edit the paths marked `# change path` / the Google Drive paths to match your setup
(dataset zip, saved weights, test folder). Use a **GPU runtime** (Runtime → Change runtime type → GPU).

Expected dataset layout (inside the zip):
```
dataset/
├── gun/  knife/  shuriken/  safe/      # images
└── annotations/
    ├── gun/  knife/  shuriken/         # segmentation masks (same base filename as the image)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q "/content/drive/MyDrive/ProjectYasminn.zip" -d "/content/ProjectYasmin"

In [ ]:
import os
import random
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
# ============================================
# U-Net Preprocessing Visualization
# ============================================

import cv2
import matplotlib.pyplot as plt

# image path
img_path = "/content/ProjectYasmin/ProjectYasminn/dataset/gun/P00103.png"   # change path

# read grayscale image
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

# preprocessing stages
resized = cv2.resize(img, (256, 256))

gaussian = cv2.GaussianBlur(resized, (3,3), 0)

normalized = gaussian.astype("float32") / 255.0

# display
titles = [
    "Original Image",
    "Resized Image",
    "Gaussian Blur",
    "Normalized Image"
]

images = [
    img,
    resized,
    gaussian,
    normalized
]

plt.figure(figsize=(12,4))

for i in range(len(images)):
    plt.subplot(1,4,i+1)
    plt.imshow(images[i], cmap='gray')
    plt.title(titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

if device == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
class BaggageBinaryDataset(Dataset):
    def __init__(self, root, img_size=256, augment=True):
        self.root = root
        self.img_size = img_size
        self.augment = augment
        self.classes = ["gun", "knife", "shuriken"]
        self.samples = []

        for cls in self.classes:
            img_dir = os.path.join(root, cls)
            mask_dir = os.path.join(root, "annotations", cls)

            image_paths = []
            for ext in ["*.png", "*.jpg", "*.jpeg"]:
                image_paths.extend(glob(os.path.join(img_dir, ext)))

            for img_path in image_paths:
                base = os.path.splitext(os.path.basename(img_path))[0]

                possible_masks = [
                    os.path.join(mask_dir, base + ".png"),
                    os.path.join(mask_dir, base + ".jpg"),
                    os.path.join(mask_dir, base + ".jpeg")
                ]

                mask_path = None

                for p in possible_masks:
                    if os.path.exists(p):
                        mask_path = p
                        break

                if mask_path is not None:
                    self.samples.append((img_path, mask_path))

        print("Matched image-mask pairs:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def preprocess_image(self, img):
        img = cv2.resize(img, (self.img_size, self.img_size))

        # Mild denoising only
        img = cv2.GaussianBlur(img, (3,3), 0)

        img = img.astype(np.float32) / 255.0

        return img

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise FileNotFoundError(img_path)

        if mask is None:
            raise FileNotFoundError(mask_path)

        img = self.preprocess_image(img)

        mask = cv2.resize(
            mask,
            (self.img_size, self.img_size),
            interpolation=cv2.INTER_NEAREST
        )

        mask = (mask > 0).astype(np.uint8)

        if self.augment:

            if random.random() > 0.5:
                img = np.fliplr(img).copy()
                mask = np.fliplr(mask).copy()

            angle = random.uniform(-20, 20)
            M = cv2.getRotationMatrix2D(
                (self.img_size/2, self.img_size/2),
                angle,
                1.0
            )

            img = cv2.warpAffine(
                img,
                M,
                (self.img_size, self.img_size),
                flags=cv2.INTER_LINEAR
            )

            mask = cv2.warpAffine(
                mask,
                M,
                (self.img_size, self.img_size),
                flags=cv2.INTER_NEAREST
            )

            alpha = random.uniform(0.9, 1.1)
            beta = random.uniform(-0.05, 0.05)
            img = np.clip(alpha * img + beta, 0, 1)

        img = np.expand_dims(img, axis=0)

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        )

In [ ]:
dataset_path = "/content/ProjectYasmin/ProjectYasminn/dataset"

full_dataset = BaggageBinaryDataset(
    dataset_path,
    img_size=256,
    augment=True
)

train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_ds, val_ds = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(1)
)

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2
)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)


class UNetBinary(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(1, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.bridge = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv1 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        b = self.bridge(p3)

        u3 = self.up3(b)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.conv3(u3)

        u2 = self.up2(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.conv1(u1)

        return self.out(u1)

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        bce = self.bce(pred, target)

        pred = torch.sigmoid(pred)

        smooth = 1e-6

        pred_flat = pred.view(-1)
        target_flat = target.view(-1)

        intersection = (pred_flat * target_flat).sum()

        dice = (2 * intersection + smooth) / (
            pred_flat.sum() + target_flat.sum() + smooth
        )

        dice_loss = 1 - dice

        return bce + dice_loss

In [ ]:
model = UNetBinary().to(device)

criterion = DiceBCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
def compute_metrics_from_logits(logits, target):
    probs = torch.sigmoid(logits)
    pred = (probs > 0.5).float()

    pred_np = pred.cpu().numpy()
    target_np = target.cpu().numpy()

    pixel_acc = (pred_np == target_np).sum() / target_np.size

    P = pred_np == 1
    G = target_np == 1

    intersection = np.logical_and(P, G).sum()
    union = np.logical_or(P, G).sum()

    iou = intersection / (union + 1e-8)

    dice = (2 * intersection) / (
        P.sum() + G.sum() + 1e-8
    )

    return pixel_acc, iou, dice

In [ ]:
epochs = 30
best_iou = 0
import random

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):

        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()

    val_loss = 0
    acc_list = []
    iou_list = []
    dice_list = []

    with torch.no_grad():
        for imgs, masks in val_loader:

            imgs = imgs.to(device)
            masks = masks.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, masks)

            val_loss += loss.item()

            acc, iou, dice = compute_metrics_from_logits(outputs, masks)

            acc_list.append(acc)
            iou_list.append(iou)
            dice_list.append(dice)

    val_loss /= len(val_loader)

    mean_acc = np.mean(acc_list)
    mean_iou = np.mean(iou_list)
    mean_dice = np.mean(dice_list)

    print(f"""
Epoch {epoch+1}/{epochs}
Train Loss : {train_loss:.4f}
Val Loss   : {val_loss:.4f}
Pixel Acc  : {mean_acc:.4f}
Mean IoU   : {mean_iou:.4f}
Mean Dice  : {mean_dice:.4f}
""")

    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/best_binary_unet_baggage_v2.pth"
        )
        print("Best model saved.")

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/best_binary_unet_baggage_v2.pth",
        map_location=device
    )
)

model.eval()

In [ ]:
img_path = "/content/ProjectYasmin/ProjectYasminn/dataset/knife/P00234.png"

def predict_mask(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    original = cv2.imread(img_path)

    if img is None:
        raise FileNotFoundError(img_path)

    img = cv2.resize(img, (256,256))
    img = cv2.GaussianBlur(img, (3,3), 0)
    img = img.astype(np.float32) / 255.0

    x = torch.tensor(img).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        out = model(x)
        prob = torch.sigmoid(out).squeeze().cpu().numpy()

    pred = (prob > 0.5).astype(np.uint8)

    return original, pred, prob


original, pred, prob = predict_mask(img_path)
gt_path = "/content/ProjectYasmin/ProjectYasminn/dataset/annotations/knife/P00234.png"
# # Ground truth path
# if "/dataset/gun/" in img_path:
#     gt_path = img_path.replace("/dataset/gun/", "/dataset/annotations/gun/")
# elif "/dataset/knife/" in img_path:
#     gt_path = img_path.replace("/dataset/knife/", "/dataset/annotations/knife/")
# elif "/dataset/shuriken/" in img_path:
#     gt_path = img_path.replace("/dataset/shuriken/", "/dataset/annotations/shuriken/")
# else:
#     raise ValueError("Unknown class folder.")

GT = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)

if GT is None:
    raise FileNotFoundError(gt_path)

GT = cv2.resize(GT, (256,256), interpolation=cv2.INTER_NEAREST)
GTmask = (GT > 0).astype(np.uint8)
GTdisplay = GTmask * 255

# ---------------- POST-PROCESSING ----------------

mask = pred.copy()

kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))

mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
    mask,
    connectivity=8
)

filtered_mask = np.zeros_like(mask)

H, W = mask.shape

for i in range(1, num_labels):

    x, y, w, h, area = stats[i]

    if area < 120:
        continue

    if area > 0.50 * H * W:
        continue

    aspect_ratio = w / (h + 1e-8)

    if aspect_ratio < 0.10 or aspect_ratio > 10:
        continue

    filtered_mask[labels == i] = 1

# ---------------- EVALUATION ----------------

intersection = np.logical_and(filtered_mask == 1, GTmask == 1).sum()
union = np.logical_or(filtered_mask == 1, GTmask == 1).sum()

IoU = intersection / (union + 1e-8)

Dice = (2 * intersection) / (
    (filtered_mask == 1).sum() + (GTmask == 1).sum() + 1e-8
)

print(f"IoU  = {IoU:.4f}")
print(f"Dice = {Dice:.4f}")

# ---------------- DISPLAY ----------------

plt.figure(figsize=(14,8))

plt.subplot(2,2,1)
plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
plt.title("Original Image")
plt.axis("off")

plt.subplot(2,2,2)
plt.imshow(filtered_mask * 255, cmap="gray", vmin=0, vmax=255)
plt.title("Post-processed Predicted Mask")
plt.axis("off")

plt.subplot(2,2,3)
plt.imshow(GTdisplay, cmap="gray", vmin=0, vmax=255)
plt.title("Ground Truth Annotation Mask")
plt.axis("off")

plt.subplot(2,2,4)
img_box = cv2.resize(cv2.cvtColor(original, cv2.COLOR_BGR2RGB), (256,256))
plt.imshow(img_box)
plt.title("Predicted Bounding Boxes")
plt.axis("off")

num_labels2, labels2, stats2, _ = cv2.connectedComponentsWithStats(
    filtered_mask,
    connectivity=8
)

for i in range(1, num_labels2):
    x, y, w, h, area = stats2[i]

    if area >= 120:
        rect = plt.Rectangle(
            (x,y),
            w,
            h,
            fill=False,
            edgecolor="red",
            linewidth=2
        )

        plt.gca().add_patch(rect)

plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm

# =========================================================
# CONFIGURATION
# =========================================================

IMAGE_SIZE = 256
THRESHOLD = 0.5

test_folder = "/content/drive/MyDrive/testImages"

output_folder = "/content/drive/MyDrive/predictedMasks"
os.makedirs(output_folder, exist_ok=True)

# =========================================================
# PREDICTION FUNCTION
# =========================================================

def predict_mask(img_path):

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")

    original_img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))

    # preprocessing
    img = cv2.GaussianBlur(original_img, (3, 3), 0)

    img = img.astype(np.float32) / 255.0

    x = torch.tensor(img).unsqueeze(0).unsqueeze(0).float().to(device)

    model.eval()

    with torch.no_grad():

        out = model(x)

        prob = torch.sigmoid(out).squeeze().cpu().numpy()

    pred = (prob > THRESHOLD).astype(np.uint8)

    return original_img, pred

# =========================================================
# POST-PROCESSING
# =========================================================

def postprocess_mask(pred):

    mask = pred.copy().astype(np.uint8)

    kernel_open = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3, 3)
    )

    kernel_close = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5)
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel_open
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel_close
    )

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask,
        connectivity=8
    )

    filtered_mask = np.zeros_like(mask)

    H, W = mask.shape

    for i in range(1, num_labels):

        x, y, w, h, area = stats[i]

        # remove tiny blobs
        if area < 120:
            continue

        # remove huge regions
        if area > 0.50 * H * W:
            continue

        aspect_ratio = w / (h + 1e-8)

        if aspect_ratio < 0.10 or aspect_ratio > 10:
            continue

        filtered_mask[labels == i] = 1

    return filtered_mask

# =========================================================
# TEST LOOP
# =========================================================

image_paths = []

for ext in ["*.png", "*.jpg", "*.jpeg"]:

    image_paths.extend(
        glob(os.path.join(test_folder, ext))
    )

print(f"Found {len(image_paths)} images")

for img_path in tqdm(image_paths):

    try:

        original_img, pred = predict_mask(img_path)

        pred = postprocess_mask(pred)

        filename = os.path.basename(img_path)
        name, _ = os.path.splitext(filename)

        # save predicted mask
        save_path = os.path.join(
            output_folder,
            name + "_mask.png"
        )

        cv2.imwrite(save_path, pred * 255)

        # =================================================
        # SHOW RESULTS
        # =================================================

        plt.figure(figsize=(8,4))

        plt.subplot(1,2,1)
        plt.imshow(original_img, cmap="gray")
        plt.title("Input Image")
        plt.axis("off")

        plt.subplot(1,2,2)
        plt.imshow(pred, cmap="gray")
        plt.title("Predicted Mask")
        plt.axis("off")

        plt.show()

    except Exception as e:

        print(f"Error with {img_path}")
        print(e)

print("\n===================================")
print("Prediction completed successfully")
print("===================================")

print(f"Predicted masks saved in:\n{output_folder}")